In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/used_cars_cleaned.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (61056, 16)


,name,year,fuel,transmission,registration_location,color,assembly,body_type,price_pkr,mileage_km,engine_cc,battery_kwh,feature_count,brand,model,vehicle_age
0,Suzuki Alto VXL AGS 2022,2022,Petrol,Automatic,Islamabad,Solid White,Local,Hatchback,2390000.0,120000.0,660.0,NaN,10,Suzuki,Alto,3
1,Honda N Box Custom GL 2022,2022,Petrol,Automatic,Islamabad,White,Imported,Hatchback,3340000.0,37110.0,658.0,NaN,20,Honda,N,3
2,Honda Civic EX 1995,1995,LPG,Manual,Karachi,Black,Local,Sedan,630000.0,786.0,1500.0,NaN,8,Honda,Civic,30
3,Toyota Corolla GLi Automatic 1.6 VVTi 2012,2012,Petrol,Automatic,Lahore,Medium Silver,Imported,Sedan,3325000.0,133000.0,1600.0,NaN,9,Toyota,Corolla,13
4,Toyota Corolla Hatchback 1998,1998,Petrol,Automatic,Punjab,Black,Local,Unknown,1645000.0,125225.0,1600.0,NaN,15,Toyota,Corolla,27


In [2]:
REFERENCE_YEAR = 2025

df["vehicle_age"] = REFERENCE_YEAR - df["year"]

print(df["vehicle_age"].describe())

df[["year", "vehicle_age"]].head(10)

count    61056.000000
mean        12.310223
std          8.976818
min          0.000000
25%          5.000000
50%         10.000000
75%         18.000000
max         73.000000
Name: vehicle_age, dtype: float64


,year,vehicle_age
0,2022,3
1,2022,3
2,1995,30
3,2012,13
4,1998,27
5,2024,1
6,2022,3
7,2021,4
8,2022,3
9,2006,19


In [3]:
df["mileage_per_year"] = (
    df["mileage_km"] / df["vehicle_age"].clip(lower=1)
)

print(df["mileage_per_year"].describe())

df[
    ["year", "vehicle_age", "mileage_km", "mileage_per_year"]
].head(10)

count     61056.000000
mean      10101.024122
std        9041.416879
min           0.042553
25%        5350.000000
50%        8857.142857
75%       13000.000000
max      800000.000000
Name: mileage_per_year, dtype: float64


,year,vehicle_age,mileage_km,mileage_per_year
0,2022,3,120000.0,40000.000000
1,2022,3,37110.0,12370.000000
2,1995,30,786.0,26.200000
3,2012,13,133000.0,10230.769231
4,1998,27,125225.0,4637.962963
5,2024,1,4100.0,4100.000000
6,2022,3,4917.0,1639.000000
7,2021,4,75000.0,18750.000000
8,2022,3,22000.0,7333.333333
9,2006,19,180000.0,9473.684211


In [4]:
df["is_electric"] = (df["fuel"] == "Electric").astype(int)

print(df["is_electric"].value_counts())

df.loc[
    df["is_electric"] == 1,
    ["fuel", "engine_cc", "battery_kwh", "is_electric"]
].head()

is_electric
0    60585
1      471
Name: count, dtype: int64


,fuel,engine_cc,battery_kwh,is_electric
8,Electric,NaN,71.00,1
236,Electric,NaN,79.00,1
402,Electric,NaN,71.00,1
523,Electric,NaN,55.00,1
542,Electric,NaN,49.92,1


In [5]:
df["mileage_missing"] = df["mileage_km"].isna().astype(int)

print(df["mileage_missing"].value_counts())

mileage_missing
0    61056
Name: count, dtype: int64


In [6]:
df["engine_cc_missing"] = (
    df["engine_cc"].isna() &
    (df["is_electric"] == 0)
).astype(int)

print(df["engine_cc_missing"].value_counts())

engine_cc_missing
0    61055
1        1
Name: count, dtype: int64


In [7]:
feature_columns = [
    "vehicle_age",
    "mileage_km",
    "mileage_per_year",
    "fuel",
    "transmission",
    "registration_location",
    "color",
    "assembly",
    "body_type",
    "engine_cc",
    "battery_kwh",
    "feature_count",
    "brand",
    "model",
    "is_electric",
    "mileage_missing",
    "engine_cc_missing"
]

X = df[feature_columns].copy()
y = df["price_pkr"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeatures:")
print(X.columns.tolist())

X shape: (61056, 17)
y shape: (61056,)

Features:
['vehicle_age', 'mileage_km', 'mileage_per_year', 'fuel', 'transmission', 'registration_location', 'color', 'assembly', 'body_type', 'engine_cc', 'battery_kwh', 'feature_count', 'brand', 'model', 'is_electric', 'mileage_missing', 'engine_cc_missing']


In [8]:
numerical_features = [
    "vehicle_age",
    "mileage_km",
    "mileage_per_year",
    "engine_cc",
    "battery_kwh",
    "feature_count",
    "is_electric",
    "mileage_missing",
    "engine_cc_missing"
]

categorical_features = [
    "fuel",
    "transmission",
    "registration_location",
    "color",
    "assembly",
    "body_type",
    "brand",
    "model"
]

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Total:", len(numerical_features) + len(categorical_features))

Numerical features: 9
Categorical features: 8
Total: 17


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['vehicle_age', 'mileage_km',
                                  'mileage_per_year', 'engine_cc',
                                  'battery_kwh', 'feature_count', 'is_electric',
                                  'mileage_missing', 'engine_cc_missing']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['fuel', 'transmission',
                                  'registration_location', 'color', 'assembly',
                                  'body_

In [10]:
numerical_features = [
    "vehicle_age",
    "mileage_km",
    "mileage_per_year",
    "engine_cc",
    "battery_kwh",
    "feature_count",
    "is_electric",
    "mileage_missing",
    "engine_cc_missing"
]

categorical_features = [
    "fuel",
    "transmission",
    "registration_location",
    "color",
    "assembly",
    "body_type",
    "brand",
    "model"
]

In [11]:
print("Numerical:", len(numerical_features))
print("Categorical:", len(categorical_features))
print("X shape:", X.shape)
print("y shape:", y.shape)

Numerical: 9
Categorical: 8
X shape: (61056, 17)
y shape: (61056,)


In [12]:
feature_columns = [
    "vehicle_age",
    "mileage_km",
    "mileage_per_year",
    "fuel",
    "transmission",
    "registration_location",
    "color",
    "assembly",
    "body_type",
    "engine_cc",
    "battery_kwh",
    "feature_count",
    "brand",
    "model",
    "is_electric",
    "mileage_missing",
    "engine_cc_missing"
]

X = df[feature_columns].copy()
y = df["price_pkr"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (61056, 17)
y shape: (61056,)


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['vehicle_age', 'mileage_km',
                                  'mileage_per_year', 'engine_cc',
                                  'battery_kwh', 'feature_count', 'is_electric',
                                  'mileage_missing', 'engine_cc_missing']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['fuel', 'transmission',
                                  'registration_location', 'color', 'assembly',
                                  'body_

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (48844, 17)
X_test: (12212, 17)
y_train: (48844,)
y_test: (12212,)


In [15]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

baseline_model = DummyRegressor(strategy="median")

baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))
baseline_r2 = r2_score(y_test, baseline_predictions)

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)
print("Baseline R²:", baseline_r2)

Baseline MAE: 2491329.102522109
Baseline RMSE: 6171301.401630305
Baseline R²: -0.04850118953024407


In [16]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

ridge_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=1.0))
])

ridge_model.fit(X_train, y_train)

ridge_predictions = ridge_model.predict(X_test)

ridge_mae = mean_absolute_error(y_test, ridge_predictions)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_predictions))
ridge_r2 = r2_score(y_test, ridge_predictions)

print("Ridge Regression MAE:", ridge_mae)
print("Ridge Regression RMSE:", ridge_rmse)
print("Ridge Regression R²:", ridge_r2)

Ridge Regression MAE: 1983454.1838031767
Ridge Regression RMSE: 4590309.829074003
Ridge Regression R²: 0.4199046128152518


In [17]:
from sklearn.ensemble import RandomForestRegressor

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=50,
        max_depth=20,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
rf_r2 = r2_score(y_test, rf_predictions)

print("Random Forest MAE:", rf_mae)
print("Random Forest RMSE:", rf_rmse)
print("Random Forest R²:", rf_r2)

Random Forest MAE: 405902.1260974244
Random Forest RMSE: 2009358.1916006554
Random Forest R²: 0.8888447100513669


In [18]:
from sklearn.ensemble import ExtraTreesRegressor

extra_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ExtraTreesRegressor(
        n_estimators=50,
        max_depth=20,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

extra_model.fit(X_train, y_train)

extra_predictions = extra_model.predict(X_test)

extra_mae = mean_absolute_error(y_test, extra_predictions)
extra_rmse = np.sqrt(mean_squared_error(y_test, extra_predictions))
extra_r2 = r2_score(y_test, extra_predictions)

print("Extra Trees MAE:", extra_mae)
print("Extra Trees RMSE:", extra_rmse)
print("Extra Trees R²:", extra_r2)

Extra Trees MAE: 374510.64644757763
Extra Trees RMSE: 1984334.7099035096
Extra Trees R²: 0.8915960092069415


In [19]:
y_train_log = np.log1p(y_train)

rf_log_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=50,
        max_depth=20,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

rf_log_model.fit(X_train, y_train_log)

# Model predicts log prices
rf_log_predictions = rf_log_model.predict(X_test)

# Convert predictions back to PKR
rf_log_predictions_pkr = np.expm1(rf_log_predictions)

rf_log_mae = mean_absolute_error(y_test, rf_log_predictions_pkr)
rf_log_rmse = np.sqrt(
    mean_squared_error(y_test, rf_log_predictions_pkr)
)
rf_log_r2 = r2_score(y_test, rf_log_predictions_pkr)

print("Log Random Forest MAE:", rf_log_mae)
print("Log Random Forest RMSE:", rf_log_rmse)
print("Log Random Forest R²:", rf_log_r2)

Log Random Forest MAE: 395360.12938135746
Log Random Forest RMSE: 2159685.7874092096
Log Random Forest R²: 0.8715906779876752


In [20]:
results = pd.DataFrame({
    "Model": [
        "Median Baseline",
        "Ridge Regression",
        "Random Forest",
        "Extra Trees",
        "Log Random Forest"
    ],
    "MAE": [
        baseline_mae,
        ridge_mae,
        rf_mae,
        extra_mae,
        rf_log_mae
    ],
    "RMSE": [
        baseline_rmse,
        ridge_rmse,
        rf_rmse,
        extra_rmse,
        rf_log_rmse
    ],
    "R2": [
        baseline_r2,
        ridge_r2,
        rf_r2,
        extra_r2,
        rf_log_r2
    ]
})

results = results.sort_values("MAE").reset_index(drop=True)

results

,Model,MAE,RMSE,R2
0,Extra Trees,3.745106e+05,1.984335e+06,0.891596
1,Log Random Forest,3.953601e+05,2.159686e+06,0.871591
2,Random Forest,4.059021e+05,2.009358e+06,0.888845
3,Ridge Regression,1.983454e+06,4.590310e+06,0.419905
4,Median Baseline,2.491329e+06,6.171301e+06,-0.048501


In [21]:
from sklearn.model_selection import cross_val_score

rf_cv_scores = cross_val_score(
    rf_model,
    X,
    y,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

rf_cv_mae = -rf_cv_scores

print("Random Forest CV MAE scores:", rf_cv_mae)
print("Mean CV MAE:", rf_cv_mae.mean())
print("Std CV MAE:", rf_cv_mae.std())

Random Forest CV MAE scores: [531095.31255287 428415.43879311 300028.48121168]
Mean CV MAE: 419846.4108525547
Std CV MAE: 94527.03781246586


In [22]:
extra_cv_scores = cross_val_score(
    extra_model,
    X,
    y,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

extra_cv_mae = -extra_cv_scores

print("Extra Trees CV MAE scores:", extra_cv_mae)
print("Mean CV MAE:", extra_cv_mae.mean())
print("Std CV MAE:", extra_cv_mae.std())

Extra Trees CV MAE scores: [494386.80575074 390800.43198569 302137.58066494]
Mean CV MAE: 395774.9394671198
Std CV MAE: 78564.20062613169


In [23]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(
    extra_model,
    "../models/used_car_price_model.joblib"
)

print("Model saved successfully!")

Model saved successfully!


In [24]:
print(os.path.exists("../models/used_car_price_model.joblib"))

True


In [25]:
absolute_percentage_errors = (
    np.abs(y_test.values - extra_predictions) / y_test.values
) * 100

print(
    "Median percentage error:",
    np.median(absolute_percentage_errors)
)

print(
    "75th percentile error:",
    np.percentile(absolute_percentage_errors, 75)
)

Median percentage error: 6.2580988453029995
75th percentile error: 12.474588489789106


In [26]:
extra_model.fit(X, y)

joblib.dump(
    extra_model,
    "../models/used_car_price_model.joblib"
)

print("Final production model refitted and saved.")

Final production model refitted and saved.


In [27]:
df["mileage_per_year"] = np.where(
    df["vehicle_age"] > 0,
    df["mileage_km"] / df["vehicle_age"],
    df["mileage_km"]
)

In [28]:
df[
    ["year", "vehicle_age", "mileage_km", "mileage_per_year"]
].head(20)

,year,vehicle_age,mileage_km,mileage_per_year
0,2022,3,120000.0,40000.000000
1,2022,3,37110.0,12370.000000
2,1995,30,786.0,26.200000
3,2012,13,133000.0,10230.769231
4,1998,27,125225.0,4637.962963
5,2024,1,4100.0,4100.000000
6,2022,3,4917.0,1639.000000
7,2021,4,75000.0,18750.000000
8,2022,3,22000.0,7333.333333
9,2006,19,180000.0,9473.684211


In [29]:
df["mileage_per_year"].describe()

count     61056.000000
mean      10101.024122
std        9041.416879
min           0.042553
25%        5350.000000
50%        8857.142857
75%       13000.000000
max      800000.000000
Name: mileage_per_year, dtype: float64

In [30]:
print(df.shape)
print(df["mileage_km"].isna().sum())

(61056, 20)
0


In [31]:
print(df["mileage_per_year"].isna().sum())
print(df["mileage_per_year"].describe())

0
count     61056.000000
mean      10101.024122
std        9041.416879
min           0.042553
25%        5350.000000
50%        8857.142857
75%       13000.000000
max      800000.000000
Name: mileage_per_year, dtype: float64


In [ ]:
df.nlargest(10, "mileage_per_year")[
    [
        "name",
        "year",
        "vehicle_age",
        "mileage_km",
        "mileage_per_year",
        "price_pkr"
    ]
]       

,name,year,vehicle_age,mileage_km,mileage_per_year,price_pkr
57543,Suzuki Every GA 2024,2024,1,800000.0,800000.0,3050000.0
44656,Toyota Corolla Cross 1.8 HEV X 2024,2024,1,260000.0,260000.0,9400000.0
59534,Changan Alsvin 1.3L MT Comfort 2024,2024,1,260000.0,260000.0,4000000.0
58210,Suzuki Wagon R 2024,2024,1,239999.0,239999.0,3250000.0
48057,Toyota Yaris Sedan ATIV X CVT 1.5 2021,2021,4,940000.0,235000.0,4650000.0
25744,Toyota Corolla Altis Grande X CVT-i 1.8 Beige ...,2021,4,927780.0,231945.0,5950000.0
60605,Suzuki Alto VXR 2021,2021,4,900000.0,225000.0,2490000.0
48795,Suzuki Cultus Auto Gear Shift 2023,2023,2,400000.0,200000.0,4090000.0
50535,Hyundai Shehzore 2025,2025,0,200000.0,200000.0,2850000.0
19568,Suzuki Bolan VX Euro II 2021,2021,4,786786.0,196696.5,1950000.0


In [33]:
df["mileage_per_year"].quantile([
    0.90,
    0.95,
    0.99,
    0.995,
    0.999
])

0.900    18000.000000
0.950    22422.932331
0.990    35011.250000
0.995    46000.000000
0.999    98501.857778
Name: mileage_per_year, dtype: float64

In [34]:
df["price_pkr"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99, 0.995, 0.999]
)

count    6.105600e+04
mean     4.068061e+06
std      6.385759e+06
min      1.300000e+05
50%      2.700000e+06
90%      7.200000e+06
95%      1.090000e+07
99%      3.054500e+07
99.5%    4.100000e+07
99.9%    7.350000e+07
max      2.500000e+08
Name: price_pkr, dtype: float64

In [35]:
df.nlargest(10, "price_pkr")[
    ["name", "year", "mileage_km", "engine_cc", "price_pkr"]
]

,name,year,mileage_km,engine_cc,price_pkr
17453,Mercedes Benz S Class S 63 E Performance AMG 2023,2023,6000.0,4000.0,250000000.0
26346,Lamborghini Urus 2020,2020,2800.0,4000.0,200000000.0
16195,Lamborghini Urus Performante 2020,2020,15000.0,4000.0,196500000.0
22616,Mercedes Benz G Class G 63 AMG 2020,2020,5000.0,4000.0,167500000.0
24189,Mercedes Benz G Class G 63 AMG 2020,2020,27000.0,4000.0,157500000.0
88,Mercedes Benz G Class G 63 AMG 2020,2020,28000.0,4000.0,155000000.0
21471,Mercedes Benz G Class G 63 AMG 2020,2020,20000.0,4000.0,155000000.0
4682,Lexus LX Series LX 600 Ultra Luxury 2023,2023,4645.0,3500.0,150000000.0
17297,Mercedes Benz S Class S 500 4MATIC 2021,2021,30000.0,3000.0,135000000.0
27821,Toyota Land Cruiser ZX Gasoline 3.5L 2025,2025,10.0,3500.0,130000000.0


In [36]:
df["log_price"] = np.log1p(df["price_pkr"])

df[["price_pkr", "log_price"]].describe()

,price_pkr,log_price
count,6.105600e+04,61056.000000
mean,4.068061e+06,14.787481
std,6.385759e+06,0.871358
min,1.300000e+05,11.775297
25%,1.470000e+06,14.200774
50%,2.700000e+06,14.808763
75%,4.450000e+06,15.308415
max,2.500000e+08,19.336971


In [37]:
print("Original price skewness:",
      df["price_pkr"].skew())

print("Log price skewness:",
      df["log_price"].skew())

Original price skewness: 9.656918108580605
Log price skewness: 0.22589318947068907


In [38]:
categorical_cols = [
    "fuel",
    "transmission",
    "registration_location",
    "color",
    "assembly",
    "body_type",
    "brand",
    "model"
]

for col in categorical_cols:
    print(f"{col}: {df[col].nunique()} unique values")

fuel: 6 unique values
transmission: 2 unique values
registration_location: 123 unique values
color: 973 unique values
assembly: 2 unique values
body_type: 22 unique values
brand: 71 unique values
model: 460 unique values


In [39]:
df[categorical_cols].isna().sum()

fuel                     0
transmission             0
registration_location    0
color                    0
assembly                 0
body_type                0
brand                    0
model                    0
dtype: int64

In [41]:
feature_cols = [
    "vehicle_age",
    "mileage_km",
    "mileage_per_year",
    "engine_cc",
    "feature_count",
    "fuel",
    "transmission",
    "assembly",
    "body_type",
    "brand"
]

X = df[feature_cols].copy()
y = df["log_price"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nMissing values:")
print(X.isna().sum().sort_values(ascending=False))

X shape: (61056, 10)
y shape: (61056,)

Missing values:
engine_cc           472
vehicle_age           0
mileage_km            0
mileage_per_year      0
feature_count         0
fuel                  0
transmission          0
assembly              0
body_type             0
brand                 0
dtype: int64


In [42]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (48844, 10)
X_test: (12212, 10)
y_train: (48844,)
y_test: (12212,)


In [43]:
numeric_features = [
    "vehicle_age",
    "mileage_km",
    "mileage_per_year",
    "engine_cc",
    "feature_count"
]

categorical_features = [
    "fuel",
    "transmission",
    "assembly",
    "body_type",
    "brand"
]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

print("\nTraining missing values:")
print(X_train[numeric_features].isna().sum())

Numeric features: ['vehicle_age', 'mileage_km', 'mileage_per_year', 'engine_cc', 'feature_count']
Categorical features: ['fuel', 'transmission', 'assembly', 'body_type', 'brand']

Training missing values:
vehicle_age           0
mileage_km            0
mileage_per_year      0
engine_cc           373
feature_count         0
dtype: int64


In [44]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['vehicle_age', 'mileage_km',
                                  'mileage_per_year', 'engine_cc',
                                  'feature_count']),
                                ('cat',
                                 Pipeline(steps=[('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['fuel', 'transmission', 'assembly',
                                  'body_type', 'brand'])])


In [45]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)

print(linear_model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['vehicle_age', 'mileage_km',
                                                   'mileage_per_year',
                                                   'engine_cc',
                                                   'feature_count']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['fuel', 'transmission',
                                                   'assembly', 'body_type',
                                                

In [46]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    linear_model,
    X_train,
    y_train,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

mae_scores = -cv_scores

print("MAE scores:", mae_scores)
print("Mean MAE:", mae_scores.mean())
print("Std MAE:", mae_scores.std())

MAE scores: [0.20652918 0.20674048 0.20672534 0.20919403 0.21047264]
Mean MAE: 0.2079323344299288
Std MAE: 0.0016056908278638238


In [47]:
from sklearn.ensemble import RandomForestRegressor

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

print(random_forest_model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['vehicle_age', 'mileage_km',
                                                   'mileage_per_year',
                                                   'engine_cc',
                                                   'feature_count']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['fuel', 'transmission',
                                                   'assembly', 'body_type',
                                                